In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from skimage.feature import hog
from mpl_toolkits.mplot3d import Axes3D

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
!pip install opencv-python scikit-image seaborn tensorflow

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from skimage.feature import hog
from mpl_toolkits.mplot3d import Axes3D

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
!pip install opencv-python scikit-image seaborn tensorflow

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from skimage.feature import hog
from mpl_toolkits.mplot3d import Axes3D

import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
dataset_path = r"D:\Plant_Disease_Project\dataset\PlantVillageDataset\PlantVillage"

classes = os.listdir(dataset_path)

print("Classes found:", classes)

In [ ]:
images = []
labels = []

img_size = 128

for label in classes:

    if "Tomato" not in label:
        continue

    folder = os.path.join(dataset_path, label)

    if not os.path.isdir(folder):
        continue

    for img in os.listdir(folder)[:150]:

        path = os.path.join(folder, img)
        image = cv2.imread(path)

        if image is None:
            continue

        image = cv2.resize(image, (img_size, img_size))

        images.append(image)
        labels.append(label)

images = np.array(images)
labels = np.array(labels)

print("Dataset shape:", images.shape)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x=labels)

plt.title("Class Distribution of Plant Disease Dataset")
plt.xticks(rotation=45)

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

for i in range(6):

    plt.subplot(2,3,i+1)
    plt.imshow(cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB))
    plt.title(labels[i])
    plt.axis("off")

plt.show()

In [ ]:
img = cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY)

x = np.arange(0, img.shape[1])
y = np.arange(0, img.shape[0])

X, Y = np.meshgrid(x, y)

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(X, Y, img, cmap='viridis')

plt.title("3D Surface Visualization of Leaf Image")
plt.show()

In [ ]:
processed_images = []

for img in images:

    img = cv2.GaussianBlur(img,(5,5),0)
    img = img / 255.0

    processed_images.append(img)

processed_images = np.array(processed_images)

print("Preprocessing complete")

In [ ]:
plt.imshow(processed_images[0])
plt.title("Preprocessed Leaf Image")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

# Raw image
plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
plt.title("Raw Leaf Image")
plt.axis("off")

# Preprocessed image
plt.subplot(1,2,2)
plt.imshow(processed_images[0])
plt.title("Preprocessed Leaf Image")
plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

# Raw image histogram
plt.subplot(1,2,1)
plt.hist(images[0].ravel(), bins=50)
plt.title("Pixel Distribution - Raw Image")

# Preprocessed histogram
plt.subplot(1,2,2)
plt.hist(processed_images[0].ravel(), bins=50)
plt.title("Pixel Distribution - Preprocessed Image")

plt.show()

In [ ]:
img = images[200]

# convert image to HSV color space
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# range for diseased color (brown/yellow regions)
lower = np.array([10, 50, 50])
upper = np.array([35, 255, 255])

# create mask
mask = cv2.inRange(hsv, lower, upper)

# segmented image
segmented = cv2.bitwise_and(img, img, mask=mask)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original Leaf")

plt.subplot(1,3,2)
plt.imshow(mask, cmap='gray')
plt.title("Disease Mask")

plt.subplot(1,3,3)
plt.imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
plt.title("Segmented Diseased Region")

plt.show()

In [ ]:
X = processed_images.reshape(len(processed_images), -1)
y = labels

print("Feature shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=3000)

log_model.fit(X_train_scaled, y_train)

pred_log = log_model.predict(X_test_scaled)

acc_log = accuracy_score(y_test, pred_log)

print("Logistic Regression Accuracy:", acc_log)

In [ ]:
knn_model = KNeighborsClassifier()

knn_model.fit(X_train, y_train)

pred_knn = knn_model.predict(X_test)

acc_knn = accuracy_score(y_test, pred_knn)

print("KNN Accuracy:", acc_knn)

In [ ]:
dt_model = DecisionTreeClassifier()

dt_model.fit(X_train, y_train)

pred_dt = dt_model.predict(X_test)

acc_dt = accuracy_score(y_test, pred_dt)

print("Decision Tree Accuracy:", acc_dt)

In [ ]:
svm_model = SVC()

svm_model.fit(X_train, y_train)

pred_svm = svm_model.predict(X_test)

acc_svm = accuracy_score(y_test, pred_svm)

print("SVM Accuracy:", acc_svm)

In [ ]:
models = ["Logistic Regression", "KNN", "Decision Tree", "SVM"]

accuracies = [acc_log, acc_knn, acc_dt, acc_svm]

plt.figure(figsize=(8,5))
plt.bar(models, accuracies)

plt.title("Model Accuracy (Without Feature Extraction)")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
hog_features = []

for img in processed_images:

    gray = cv2.cvtColor((img*255).astype("uint8"), cv2.COLOR_BGR2GRAY)

    features = hog(
        gray,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        visualize=False
    )

    hog_features.append(features)

hog_features = np.array(hog_features)

print("HOG feature shape:", hog_features.shape)

In [ ]:
X_train_hog, X_test_hog, y_train_hog, y_test_hog = train_test_split(
    hog_features, labels, test_size=0.2, random_state=42
)

In [ ]:
log_hog = LogisticRegression(max_iter=1000)

log_hog.fit(X_train_hog, y_train_hog)

pred_log_hog = log_hog.predict(X_test_hog)

acc_log_hog = accuracy_score(y_test_hog, pred_log_hog)

print("Logistic Regression (HOG) Accuracy:", acc_log_hog)

In [ ]:
knn_hog = KNeighborsClassifier()

knn_hog.fit(X_train_hog, y_train_hog)

pred_knn_hog = knn_hog.predict(X_test_hog)

acc_knn_hog = accuracy_score(y_test_hog, pred_knn_hog)

print("KNN (HOG) Accuracy:", acc_knn_hog)

In [ ]:
dt_hog = DecisionTreeClassifier()

dt_hog.fit(X_train_hog, y_train_hog)

pred_dt_hog = dt_hog.predict(X_test_hog)

acc_dt_hog = accuracy_score(y_test_hog, pred_dt_hog)

print("Decision Tree (HOG) Accuracy:", acc_dt_hog)

In [ ]:
svm_hog = SVC()

svm_hog.fit(X_train_hog, y_train_hog)

pred_svm_hog = svm_hog.predict(X_test_hog)

acc_svm_hog = accuracy_score(y_test_hog, pred_svm_hog)

print("SVM (HOG) Accuracy:", acc_svm_hog)

In [ ]:
models = ["Logistic Regression", "KNN", "Decision Tree", "SVM"]

accuracies_hog = [acc_log_hog, acc_knn_hog, acc_dt_hog, acc_svm_hog]

plt.figure(figsize=(8,5))
plt.bar(models, accuracies_hog)

plt.title("Model Accuracy (With HOG Feature Extraction)")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

test_img_path = r"D:\Plant_Disease_Project\test_images\img4.JPG"

img = cv2.imread(test_img_path)

if img is None:
    print("Error: Image not found. Check the file path.")
else:
    
    display_img = img.copy()

    img = cv2.resize(img, (128,128))
    img = cv2.GaussianBlur(img, (5,5), 0)
    img = img / 255.0

    img_flat = img.reshape(1, -1)

    prediction = svm_model.predict(img_flat)

    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.title("Predicted Disease: " + prediction[0])
    plt.axis("off")

    print("Predicted Class:", prediction[0])

In [ ]:
img = cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY)

# Normalize
img_norm = img / 255.0

# Z-shaped function
z_img = 1 - np.exp(-5 * img_norm)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Original Gray Image")

plt.subplot(1,2,2)
plt.imshow(z_img, cmap='gray')
plt.title("Z-Function Enhanced Image")

plt.show()

In [ ]:
gray = cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY)

_, otsu_thresh = cv2.threshold(
    gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(gray, cmap='gray')
plt.title("Gray Image")

plt.subplot(1,2,2)
plt.imshow(otsu_thresh, cmap='gray')
plt.title("Otsu Segmentation")

plt.show()

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
plt.title("HSV Segmentation")

plt.subplot(1,3,2)
plt.imshow(otsu_thresh, cmap='gray')
plt.title("Otsu Segmentation")

plt.subplot(1,3,3)
plt.imshow(z_img, cmap='gray')
plt.title("Z-Function Enhancement")

plt.show()

In [ ]:
img = cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
clahe_img = clahe.apply(img)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Original Gray")

plt.subplot(1,2,2)
plt.imshow(clahe_img, cmap='gray')
plt.title("CLAHE Enhanced")

plt.show()

In [ ]:
edges = cv2.Canny(img, 100, 200)

plt.imshow(edges, cmap='gray')
plt.title("Edge Detection (Canny)")
plt.show()

In [ ]:
kernel = np.ones((3,3), np.uint8)

opening = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(opening, cmap='gray')
plt.title("Opening")

plt.subplot(1,2,2)
plt.imshow(closing, cmap='gray')
plt.title("Closing")

plt.show()

In [ ]:
plt.figure(figsize=(12,6))

plt.subplot(2,2,1)
plt.imshow(cv2.cvtColor(images[0], cv2.COLOR_BGR2RGB))
plt.title("Original Image")

plt.subplot(2,2,2)
plt.imshow(segmented)
plt.title("HSV Segmentation")

plt.subplot(2,2,3)
plt.imshow(otsu_thresh, cmap='gray')
plt.title("Otsu Segmentation")

plt.subplot(2,2,4)
plt.imshow(clahe_img, cmap='gray')
plt.title("CLAHE Enhancement")

plt.show()



In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

In [ ]:
X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
    processed_images, labels_encoded, test_size=0.2, random_state=42
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

cnn_model = models.Sequential()

cnn_model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
cnn_model.add(layers.MaxPooling2D((2,2)))

cnn_model.add(layers.Conv2D(64, (3,3), activation='relu'))
cnn_model.add(layers.MaxPooling2D((2,2)))

cnn_model.add(layers.Conv2D(128, (3,3), activation='relu'))
cnn_model.add(layers.MaxPooling2D((2,2)))

cnn_model.add(layers.Flatten())

cnn_model.add(layers.Dense(128, activation='relu'))
cnn_model.add(layers.Dense(len(classes), activation='softmax'))

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    epochs=5,
    validation_data=(X_test_cnn, y_test_cnn)
)

In [ ]:
loss, acc = cnn_model.evaluate(X_test_cnn, y_test_cnn)

print("CNN Accuracy:", acc)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

test_img_path = r"D:\Plant_Disease_Project\test_images\test_image.jpg"

img = cv2.imread(test_img_path)

if img is None:
    print("Image not found")
else:
    display_img = img.copy()

    img = cv2.resize(img, (128,128))
    img = cv2.GaussianBlur(img, (5,5), 0)
    img = img / 255.0

    img = np.expand_dims(img, axis=0)

    prediction = cnn_model.predict(img)

    predicted_class = np.argmax(prediction)
    predicted_label = encoder.inverse_transform([predicted_class])

    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.title("Predicted Disease: " + predicted_label[0])
    plt.axis("off")

    print("Predicted Class:", predicted_label[0])

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY)


img_norm = img / 255.0

mu_positive = img_norm

mu_negative = 1 - img_norm

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")

plt.subplot(1,3,2)
plt.imshow(mu_positive, cmap='gray')
plt.title("Positive Membership (Disease)")

plt.subplot(1,3,3)
plt.imshow(mu_negative, cmap='gray')
plt.title("Negative Membership (Healthy)")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

test_img_path = r"D:\Plant_Disease_Project\test_images\img6.JPG"

img = cv2.imread(test_img_path)

if img is None:
    print("Image not found")
else:
    display_img = img.copy()

    img = cv2.resize(img, (128,128))
    img = cv2.GaussianBlur(img, (5,5), 0)
    img = img / 255.0

    img = np.expand_dims(img, axis=0)

    prediction = cnn_model.predict(img)

    predicted_class = np.argmax(prediction)
    predicted_label = encoder.inverse_transform([predicted_class])

    plt.imshow(cv2.cvtColor(display_img, cv2.COLOR_BGR2RGB))
    plt.title("Predicted Disease: " + predicted_label[0])
    plt.axis("off")

    print("Predicted Class:", predicted_label[0])

In [ ]:
enhanced_images = []

for img in images:
    img = cv2.resize(img, (128,128))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray)
    img_norm = clahe_img / 255.0
    k = 5
    z_img = 1 - np.exp(-k * img_norm)
    z_img = (z_img * 255).astype(np.uint8)
    z_img = cv2.cvtColor(z_img, cv2.COLOR_GRAY2BGR)
    z_img = z_img / 255.0
    enhanced_images.append(z_img)

enhanced_images = np.array(enhanced_images)

print(enhanced_images.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

In [ ]:
from sklearn.model_selection import train_test_split

X_train1, X_test1, y_train1, y_test1 = train_test_split(
    processed_images, labels_encoded, test_size=0.2, random_state=42
)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    enhanced_images, labels_encoded, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras import layers, models

def create_cnn():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dense(len(classes), activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
cnn1 = create_cnn()

history1 = cnn1.fit(
    X_train1, y_train1,
    epochs=5,
    validation_data=(X_test1, y_test1)
)

In [ ]:
cnn2 = create_cnn()

history2 = cnn2.fit(
    X_train2, y_train2,
    epochs=5,
    validation_data=(X_test2, y_test2)
)

In [ ]:
cnn2 = create_cnn()

history2 = cnn2.fit(
    X_train2, y_train2,
    epochs=5,
    validation_data=(X_test2, y_test2)
)

In [ ]:
loss2, acc2 = cnn2.evaluate(X_test2, y_test2)

print("CNN Accuracy :", acc2)

In [ ]:
enhanced_images = []

for img in images:
    img = cv2.resize(img, (128,128))

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)

    lab = cv2.merge((l,a,b))
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    img = img / 255.0

    enhanced_images.append(img)

enhanced_images = np.array(enhanced_images)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

datagen.fit(X_train2)

In [ ]:
def create_better_cnn():
    model = models.Sequential()

    model.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(64, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Conv2D(128, (3,3), activation='relu'))
    model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())

    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))

    model.add(layers.Dense(len(classes), activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
cnn2 = create_better_cnn()

history2 = cnn2.fit(
    datagen.flow(X_train2, y_train2, batch_size=32),
    epochs=8,
    validation_data=(X_test2, y_test2)
)

In [ ]:
loss2, acc2 = cnn2.evaluate(X_test2, y_test2)

print("Improved CNN Accuracy:", acc2)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

In [ ]:
base_model = MobileNetV2(
    input_shape=(128,128,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train2, y_train2,
    epochs=3,
    validation_data=(X_test2, y_test2)
)

In [ ]:
history = model.fit(
    X_train2, y_train2,
    epochs=3,
    validation_data=(X_test2, y_test2)
)

In [ ]:
loss, acc_transfer = model.evaluate(X_test2, y_test2)

print("CNN Accuracy (After Enhancement):", acc_transfer)